In [1]:
import os
import numpy as np
import nibabel as nib
from scipy.ndimage import zoom
from tqdm import tqdm
import pandas as pd

In [2]:
# --- AYARLAR ---
SOURCE_DIR = "C:\\Users\\berkk\\Desktop\\Bitirme_Projesi\\dataset"
TARGET_DIR = "C:\\Users\\berkk\\Desktop\\Bitirme_Projesi\\processed_dataset_128"
TARGET_SIZE = (128, 128, 128)

# Hedef klasörü oluştur
os.makedirs(TARGET_DIR, exist_ok=True)
print(f"Hedef Klasör: {TARGET_DIR}")

Hedef Klasör: C:\Users\berkk\Desktop\Bitirme_Projesi\processed_dataset_128


In [3]:
def process_and_save(source_path, target_path):
    """
    Bir MRI dosyasını yükler, normalize eder, resize yapar ve kaydeder.
    """
    if os.path.exists(target_path):
        return  # Zaten işlenmişse atla

    try:
        # 1. Yükle
        img = nib.load(source_path)
        data = img.get_fdata()

        # 2. Normalizasyon (0-1 arası)
        data_min, data_max = np.min(data), np.max(data)
        if data_max > data_min:
            data = (data - data_min) / (data_max - data_min)
        else:
            data = np.zeros_like(data)
        
        data = data.astype(np.float32)

        # 3. Resize (Zoom)
        # Mevcut boyutlar
        d, h, w = data.shape
        
        # Zoom faktörleri
        zoom_factors = (
            TARGET_SIZE[0] / d,
            TARGET_SIZE[1] / h,
            TARGET_SIZE[2] / w,
        )
        
        # Resize işlemi (Order=1: Bilinear interpolation, hızlıdır)
        data_resized = zoom(data, zoom_factors, order=1)
        
        # 4. Kaydet (.npy olarak)
        np.save(target_path, data_resized)
        
    except Exception as e:
        print(f"Hata oluştu: {source_path} -> {e}")

In [4]:
def scan_and_process(root_dir):
    """
    Dataset klasörünü tarar ve tüm uygun dosyaları işler.
    """
    # Train ve Test klasörlerini gez
    for split in ["Train", "Test"]:
        split_path = os.path.join(root_dir, split)
        if not os.path.exists(split_path):
            print(f"{split} klasörü bulunamadı, atlanıyor.")
            continue
            
        print(f"Processing {split} set...")
        
        # Sınıf klasörleri (0, 0.5, 1, 2)
        class_dirs = os.listdir(split_path)
        for cdir in class_dirs:
            class_path = os.path.join(split_path, cdir)
            if not os.path.isdir(class_path):
                continue
                
            # Session klasörleri
            sessions = os.listdir(class_path)
            # TQDM ile ilerleme çubuğu göster
            for sess in tqdm(sessions, desc=f"Class {cdir}"):
                sess_path = os.path.join(class_path, sess)
                if not os.path.isdir(sess_path):
                    continue
                
                # MRI dosyasını bul (T1w içeren dosya)
                mri_files = os.listdir(sess_path)
                t1_file = next((x for x in mri_files if "T1w" in x), None)
                
                if t1_file:
                    source_file_path = os.path.join(sess_path, t1_file)
                    
                    # Hedef dosya ismi: SessionID.npy (Örn: OAS30001_MR_d0129.npy)
                    # Bu sayede dosya isminden veriyi bulabiliriz
                    target_file_name = f"{sess}.npy"
                    target_file_path = os.path.join(TARGET_DIR, target_file_name)
                    
                    process_and_save(source_file_path, target_file_path)

In [5]:
# --- İŞLEMİ BAŞLAT ---
print("Ön işleme başlıyor... Bu işlem veri boyutuna göre zaman alabilir.")
scan_and_process(SOURCE_DIR)
print("\nİşlem tamamlandı! Tüm dosyalar .npy formatında kaydedildi.")

Ön işleme başlıyor... Bu işlem veri boyutuna göre zaman alabilir.
Processing Train set...


Class 2: 100%|██████████| 18/18 [00:10<00:00,  1.64it/s]


Processing Test set...


Class 2: 100%|██████████| 8/8 [00:06<00:00,  1.19it/s]


İşlem tamamlandı! Tüm dosyalar .npy formatında kaydedildi.
